This notebook runs skeletonization on segmentation in a zarr format using the tensorstore package.

In [2]:
import skimage
import kimimaro
import cloudvolume
from cloudvolume import Skeleton
from scipy.spatial import KDTree
from joblib import Parallel, delayed, parallel_config
import itertools
import scipy
import tarfile
from io import BytesIO
import concurrent.futures
import navis
import tarfile
from io import BytesIO
import concurrent.futures
import itertools
import numpy as np
from kimimaro.intake import merge

from ac_segmentation.neurotorch.datasets.dataset import open_ZarrTensor, create_EmptyTensor

In [3]:
def label_binary_array(binary_arr, size_threshold=10):
    struct = scipy.ndimage.generate_binary_structure(3, 3)
    labeled_arr = np.empty(binary_arr.shape, dtype=np.uint32)
    num_features = scipy.ndimage.label(binary_arr, structure=struct, output=labeled_arr)

    if num_features > 1:
        labeled_arr = skimage.morphology.remove_small_objects(
            labeled_arr, min_size=size_threshold, connectivity=3, out=labeled_arr)

    return labeled_arr, num_features

def threshold_binarize_array(arr, threshold=0.05):
    return (arr >= threshold)

def skeletonize(out_arr, probability_threshold=0.05, label_size_threshold=50, scale=2, constant=5, 
                fill_holes=False, parallel=5, dust_threshold=10):
    # binarize volume, label, and skeletonize
    binary_arr = threshold_binarize_array(out_arr, threshold=probability_threshold)
    labeled_arr, num_feat = label_binary_array(binary_arr, size_threshold=label_size_threshold)
    skels = kimimaro.skeletonize(
        labeled_arr,
        teasar_params={
            "scale": scale, 
            "const": constant, # influences the finger branches allowed
            "pdrf_scale": 10000,
            "pdrf_exponent": 1,
            "soma_acceptance_threshold": 3500, # physical units
            "soma_detection_threshold": 750, # physical units
            "soma_invalidation_const": 300, # physical units
            "soma_invalidation_scale": 2,
            "max_paths": 50, # default None
        },
        dust_threshold=dust_threshold, # skip connected components with fewer than this many voxels
        anisotropy=(1,1,1), # default True #influences the dimension scale
        fix_branching=True, # default True
        fix_borders=True, # default True
        fill_holes=fill_holes, # default False
        fix_avocados=False, # default False
        progress=False, # default False, show progress bar
        parallel=parallel, # <= 0 all cpu, 1 single process, 2+ multiprocess
        parallel_chunk_size=100, # how many skeletons to process before updating progress bar
    )

    return skels

In [4]:
def TS_skeletonize_volume(seg_arr, chunk_size=[1000,1000,1000], n_jobs=4):
    def skel_chunk(start, end):
        skels = skeletonize(np.array(seg_arr[start[0]:end[0],start[1]:end[1],start[2]:end[2]]))

        if len(skels) != 0:
            skels = [skel for skid,skel in skels.items()]
            skels = Skeleton.simple_merge(skels).consolidate()
            skels.vertices += start
            return skels

    dx,dy,dz = seg_arr.shape
    xch, ych, zch = chunk_size
    sind_x, sind_y, sind_z = list(range(0,dx,xch)), list(range(0,dy,ych)), list(range(0,dz,zch))
    eind_x, eind_y, eind_z = [x + xch for x in sind_x], [x + ych for x in sind_y], [x + zch for x in sind_z]
    eind_x, eind_y, eind_z = [dx if ele > dx else ele for ele in eind_x], [dy if ele > dy else ele for ele in eind_y], [dz if ele > dz else ele for ele in eind_z]
    comb1 = list(itertools.product(sind_x,sind_y,sind_z))
    comb2 = list(itertools.product(eind_x,eind_y,eind_z))
    del sind_x, sind_y, sind_z, eind_x, eind_y, eind_z

    with parallel_config(backend="loky", inner_max_num_threads=2):
        results = Parallel(n_jobs=n_jobs)(delayed(skel_chunk)(x, y) for x, y in zip(comb1,comb2))
    out_skels = [i for i in results if i is not None]
    
    return results

def join_components(skels, radius = 2):
    #log which chunks the skeletons came from
    total_skels = []
    chunk_ind = []
    for ind,sk in enumerate(skels):
        temp_skels = sk.components()
        total_skels += temp_skels
        chunk_ind += len(temp_skels) * [ind]
        
    #extract root node coords
    root_vts = [] #node coordinates
    root_ind = [] #skeleton number from total_skels list
    root_node = [] #node associated with root_vt
    for ind,sk in enumerate(total_skels):  
        t_ids = sk.terminals()
        root_node += list(t_ids)
        ends = [sk.vertices[i] for i in t_ids]
        for end in ends:
            root_vts.append(list(end))
            root_ind.append(ind)
    
    #create kdtree and find all end nodes within given radius
    tree = KDTree(root_vts, leafsize=2)
    pairs = tree.query_pairs(radius)
    
    #get merge pair indices
    merge_pairs = []
    merge_pairs_vts = []
    for p1,p2 in pairs:
        merge_pairs.append([root_ind[p1], root_ind[p2]])
        merge_pairs_vts.append([root_vts[p1], root_vts[p2]])
    
    #check if skeletons in same chunk, if not merge
    t = 0
    fused_skels = []
    for ind, (m1,m2) in enumerate(merge_pairs):
        if chunk_ind[m1] == chunk_ind[m2]:
            continue
        else:
            try:
                fused = total_skels[m1].merge(total_skels[m2])
                v1, v2 = merge_pairs_vts[ind]
                n1, n2 = fused.vertices.tolist().index(v1), fused.vertices.tolist().index(v2)
                fused.edges = np.append(fused.edges, np.array([n1,n2]).reshape((1, 2)), axis=0)
                total_skels[m1] = None
                total_skels[m2] = None
                fused_skels.append(fused)
            except:
                pass
                
    out_skels = [i for i in total_skels if i is not None] + [i for i in fused_skels if i is not None]
    return out_skels

In [5]:
def write_kimi_skels_tar(tar_fn, skels):
    with tarfile.open(tar_fn, mode="w:gz") as t:
        id = 1
        for skel in skels:
            bio = BytesIO(skel.to_swc().encode())
            info = tarfile.TarInfo(name=f"{id}.swc")
            info.size = len(bio.getbuffer())
            t.addfile(tarinfo=info, fileobj=bio)
            id += 1

def read_navis_neurons_tar(tar_fn, concurrency=10, preprocess_func=None):
    preprocess_func = ((lambda x: x) if preprocess_func is None else preprocess_func)
    with concurrent.futures.ProcessPoolExecutor(max_workers=concurrency) as e:
        futs = []
        with tarfile.open(tar_fn, "r:gz") as t:
            for m in t.getmembers():
                swc_b = t.extractfile(m).read()
                futs.append(e.submit(navis.io.read_swc, swc_b.decode()))
        navis_neurons = navis.NeuronList([
            preprocess_func(fut.result()) for
            fut in concurrent.futures.as_completed(futs)])
    return navis_neurons

In [7]:
###skeletonize and join chunks with tensorstore array

arr = open_ZarrTensor('/ACdata/Users/connorl/Test_Array.zarr', bytes_limit= 100_000_000)
%time skels = TS_skeletonize_volume(arr, chunk_size=[288,288,1000], n_jobs=20)
%time joined_skels = join_components(skels, radius = 50)

CPU times: user 1.72 s, sys: 10.1 s, total: 11.8 s
Wall time: 47.2 s
CPU times: user 834 ms, sys: 122 µs, total: 834 ms
Wall time: 831 ms


In [8]:
#write neurons to tar file as swcs
write_kimi_skels_tar("/ACdata/Users/connorl/testskels.swc.gz", joined_skels)

In [9]:
#read and visualize neurons
neu = read_navis_neurons_tar("/ACdata/Users/connorl/testskels.swc.gz")
neu.plot3d(backend='plotly') 